# Compare Original vs Optimized to_dataframes.py Outputs

This notebook compares the outputs from the original and optimized versions of `to_dataframes.py` to verify they produce identical results.

We'll compare:
- `truth_reco_match_n10.csv` (original)
- `truth_reco_match_n10_opt.csv` (optimized with `--tag opt`)


In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")


Libraries imported successfully!


## 1. Load the CSV Files

Load both the original and optimized outputs.


In [2]:
# Define file paths - adjust if needed
output_dir = "/global/homes/j/jvmead/dune/light_file_processing/outputs/mr6-5_max_distance/test/"
original_file = f"{output_dir}truth_reco_match_n10.csv"
optimized_file = f"{output_dir}truth_reco_match_n10_opt_fixed.csv"

# Load the datasets
print("Loading original file...")
df_original = pd.read_csv(original_file)
print(f"Original: {len(df_original)} rows, {len(df_original.columns)} columns")

print("\nLoading optimized file...")
df_optimized = pd.read_csv(optimized_file)
print(f"Optimized: {len(df_optimized)} rows, {len(df_optimized.columns)} columns")

print("\n✓ Files loaded successfully!")


Loading original file...
Original: 25204 rows, 85 columns

Loading optimized file...
Optimized: 20924 rows, 85 columns

✓ Files loaded successfully!
Optimized: 20924 rows, 85 columns

✓ Files loaded successfully!


In [ ]:
# Compare unique (file_id, event_id, tpc_num) groups in both outputs
orig_groups = set(tuple(x) for x in df_original[['file_id', 'event_id', 'tpc_num']].drop_duplicates().values)
opt_groups = set(tuple(x) for x in df_optimized[['file_id', 'event_id', 'tpc_num']].drop_duplicates().values)

missing_in_opt = orig_groups - opt_groups
missing_in_orig = opt_groups - orig_groups

print(f"Unique (file_id, event_id, tpc_num) in original:  {len(orig_groups)}")
print(f"Unique (file_id, event_id, tpc_num) in optimized: {len(opt_groups)}")
print(f"Missing in optimized: {len(missing_in_opt)}")
print(f"Missing in original: {len(missing_in_orig)}")

if missing_in_opt:
    print("\nExamples missing in optimized:")
    for tup in list(missing_in_opt)[:10]:
        print(tup)

if missing_in_orig:
    print("\nExamples missing in original:")
    for tup in list(missing_in_orig)[:10]:
        print(tup)

In [ ]:
# For a few missing groups, print all rows from both dataframes for detailed inspection
print("\n--- Detailed comparison for missing groups (up to 5 examples) ---")
for tup in list(missing_in_opt)[:5]:
    print(f"\nGroup missing in optimized: {tup}")
    print("Original rows:")
    display(df_original[(df_original['file_id'] == tup[0]) & (df_original['event_id'] == tup[1]) & (df_original['tpc_num'] == tup[2])])
    print("Optimized rows:")
    display(df_optimized[(df_optimized['file_id'] == tup[0]) & (df_optimized['event_id'] == tup[1]) & (df_optimized['tpc_num'] == tup[2])])

for tup in list(missing_in_orig)[:5]:
    print(f"\nGroup missing in original: {tup}")
    print("Optimized rows:")
    display(df_optimized[(df_optimized['file_id'] == tup[0]) & (df_optimized['event_id'] == tup[1]) & (df_optimized['tpc_num'] == tup[2])])
    print("Original rows:")
    display(df_original[(df_original['file_id'] == tup[0]) & (df_original['event_id'] == tup[1]) & (df_original['tpc_num'] == tup[2])])

## 2. Compare Data Structures

Check if the dataframes have the same shape, columns, and data types.


In [ ]:
# Compare shapes
print("=" * 80)
print("SHAPE COMPARISON")
print("=" * 80)
print(f"Original shape:  {df_original.shape}")
print(f"Optimized shape: {df_optimized.shape}")
if df_original.shape == df_optimized.shape:
    print("✓ Shapes match!")
else:
    print("✗ WARNING: Shapes differ!")

# Compare columns
print("\n" + "=" * 80)
print("COLUMN COMPARISON")
print("=" * 80)
original_cols = set(df_original.columns)
optimized_cols = set(df_optimized.columns)

if original_cols == optimized_cols:
    print(f"✓ Both have {len(original_cols)} columns")
else:
    missing_in_opt = original_cols - optimized_cols
    missing_in_orig = optimized_cols - original_cols
    if missing_in_opt:
        print(f"✗ Missing in optimized: {missing_in_opt}")
    if missing_in_orig:
        print(f"✗ Missing in original: {missing_in_orig}")

# Show first few columns
print(f"\nFirst 10 columns: {list(df_original.columns[:10])}")


## 3. Display Basic Statistics

Compare summary statistics for key columns.


In [ ]:
# Select key columns for comparison
key_columns = ['file_id', 'event_id', 'tpc_num', 'n_photons', 'dE_tot',
               'x_mean', 'y_mean', 'z_mean', 'enu', 'n_int_per_tpc']

# Filter to columns that exist
key_columns = [col for col in key_columns if col in df_original.columns]

print("=" * 80)
print("ORIGINAL DATA STATISTICS")
print("=" * 80)
print(df_original[key_columns].describe())

print("\n" + "=" * 80)
print("OPTIMIZED DATA STATISTICS")
print("=" * 80)
print(df_optimized[key_columns].describe())


## 4. Compare Detector Hit Columns

Compare the detector columns (`det_0` through `det_15`).


In [ ]:
# Count detector hits
det_cols = [f'det_{i}' for i in range(16) if f'det_{i}' in df_original.columns]

print("=" * 80)
print("DETECTOR HIT COUNTS")
print("=" * 80)
print(f"\nOriginal vs Optimized detector hits:")
for det_col in det_cols:
    count_orig = df_original[det_col].sum()
    count_opt = df_optimized[det_col].sum()
    match = "✓" if count_orig == count_opt else "✗"
    print(f"  {det_col}: {count_orig:6.0f} (orig) vs {count_opt:6.0f} (opt) {match}")

# Total hits across all detectors
total_orig = sum(df_original[col].sum() for col in det_cols)
total_opt = sum(df_optimized[col].sum() for col in det_cols)
print(f"\nTotal hits: {total_orig:.0f} (orig) vs {total_opt:.0f} (opt)")
if total_orig == total_opt:
    print("✓ Total detector hits match!")
else:
    print(f"✗ Difference: {abs(total_orig - total_opt):.0f} hits")


## 5. Visualize Key Distributions

Create comparison histograms for important physics variables.


In [ ]:
# Create comparison plots
columns_to_plot = ['n_photons', 'dE_tot', 'enu', 'x_mean', 'y_mean', 'z_mean']
columns_to_plot = [col for col in columns_to_plot if col in df_original.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(columns_to_plot):
    ax = axes[idx]

    # Remove NaN values for cleaner plots
    orig_data = df_original[col].dropna()
    opt_data = df_optimized[col].dropna()

    # Plot overlaid histograms
    ax.hist(orig_data, bins=50, alpha=0.5, label='Original', color='blue', density=True)
    ax.hist(opt_data, bins=50, alpha=0.5, label='Optimized', color='red', density=True)

    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.set_title(f'{col} Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_histograms.png', dpi=150, bbox_inches='tight')
print("Saved comparison_histograms.png")
plt.show()


## 6. Calculate Element-wise Differences

For rows that should match, calculate the differences in numerical columns.


In [ ]:
# Sort both dataframes by key columns for alignment
sort_cols = ['file_id', 'event_id', 'tpc_num', 'vertex_id']
sort_cols = [col for col in sort_cols if col in df_original.columns]

if len(df_original) == len(df_optimized):
    df_orig_sorted = df_original.sort_values(by=sort_cols).reset_index(drop=True)
    df_opt_sorted = df_optimized.sort_values(by=sort_cols).reset_index(drop=True)

    # Calculate differences for numerical columns
    numerical_cols = df_original.select_dtypes(include=[np.number]).columns

    print("=" * 80)
    print("NUMERICAL DIFFERENCES (after sorting)")
    print("=" * 80)

    differences = {}
    for col in numerical_cols:
        if col in df_opt_sorted.columns:
            diff = df_orig_sorted[col] - df_opt_sorted[col]
            non_zero_diff = diff[diff != 0].dropna()

            if len(non_zero_diff) > 0:
                differences[col] = {
                    'count': len(non_zero_diff),
                    'max_diff': non_zero_diff.abs().max(),
                    'mean_diff': non_zero_diff.mean()
                }
                print(f"\n{col}:")
                print(f"  Non-zero differences: {len(non_zero_diff)}")
                print(f"  Max absolute diff: {non_zero_diff.abs().max():.6f}")
                print(f"  Mean diff: {non_zero_diff.mean():.6f}")

    if not differences:
        print("\n✓ All numerical values match perfectly!")
    else:
        print(f"\n✗ Found differences in {len(differences)} columns")
else:
    print(f"Cannot compare element-wise: different number of rows")
    print(f"Original: {len(df_original)}, Optimized: {len(df_optimized)}")


## 7. Compare Detector Max Values

Check if the detector max values (e.g., `det_0_max`, `det_1_max`, etc.) match.


In [ ]:
# Compare detector max values
det_max_cols = [f'det_{i}_max' for i in range(16) if f'det_{i}_max' in df_original.columns]

if len(df_original) == len(df_optimized):
    print("=" * 80)
    print("DETECTOR MAX VALUE COMPARISON")
    print("=" * 80)

    all_match = True
    for col in det_max_cols[:4]:  # Check first 4 as examples
        orig_vals = df_orig_sorted[col].dropna()
        opt_vals = df_opt_sorted[col].dropna()

        if len(orig_vals) == len(opt_vals):
            diff = (orig_vals.values - opt_vals.values)
            max_diff = np.abs(diff).max()

            if max_diff > 1e-6:
                print(f"{col}: Max difference = {max_diff:.6f}")
                all_match = False
            else:
                print(f"{col}: ✓ Match (max diff < 1e-6)")

    if all_match:
        print("\n✓ All detector max values match!")


## 8. Summary Report

Generate a final summary of the comparison.


In [ ]:
print("=" * 80)
print("FINAL COMPARISON SUMMARY")
print("=" * 80)

# Check list
checks = []
checks.append(("Shape match", df_original.shape == df_optimized.shape))
checks.append(("Column names match", set(df_original.columns) == set(df_optimized.columns)))
checks.append(("Row count match", len(df_original) == len(df_optimized)))

# Check if key statistics are similar
if len(df_original) == len(df_optimized):
    for col in key_columns:
        if col in df_original.columns:
            mean_diff = abs(df_original[col].mean() - df_optimized[col].mean())
            checks.append((f"{col} mean similar", mean_diff < 0.01))

print("\nValidation Checks:")
for check_name, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"  {check_name:30s}: {status}")

# Overall verdict
if all(result for _, result in checks):
    print("\n" + "="*80)
    print("🎉 SUCCESS: Original and Optimized outputs match!")
    print("="*80)
else:
    print("\n" + "="*80)
    print("⚠️  WARNING: Some differences detected. Review details above.")
    print("="*80)


In [ ]:
# Loop over all columns and plot histograms for each (numeric only)
import matplotlib.pyplot as plt
import numpy as np

# Find common columns
common_cols = [col for col in df_original.columns if col in df_optimized.columns]

for col in common_cols:
    # Only plot for numeric columns
    if not (np.issubdtype(df_original[col].dtype, np.number) and np.issubdtype(df_optimized[col].dtype, np.number)):
        continue
    plt.figure(figsize=(8,4))
    plt.hist(df_original[col].dropna(), bins=50, alpha=0.5, label='Original', color='C0')
    plt.hist(df_optimized[col].dropna(), bins=50, alpha=0.5, label='Optimized', color='C1')
    plt.title(f"Histogram: {col}")
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()
